# 08 — Containerize

Connect CWL execution dependencies to component image builds and distinguish tags from immutable identities.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/08-containerize/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/08-containerize/lesson.md) · [Exercise](../modules/08-containerize/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/08-containerize"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Inspect the execution dependencies

Each tool has its own `DockerRequirement`. Tags identify a requested release; a registry digest identifies exact published bytes. Inspect the Dockerfiles and compare their entrypoints with the CWL base commands.


In [ ]:
(
set -euo pipefail
python "$WORKSPACE/scripts/supply_chain.py" inventory --contract "$WORKSPACE/reference/waterbodies.cwl"
for component in crop norm-diff otsu stac; do
    printf '\nDockerfile for %s\n' "$component"
    cat "$WORKSPACE/reference/containers/$component/Dockerfile"
done
)


## Build and run images when Docker is available

A Kubernetes-backed editor does not automatically provide a Docker engine. Set `LAB_RUN_CONTAINERS=1` in the next cell only in a workspace with a working engine; the enabled path checks it and builds the four real images. The default records a skipped container exercise. Complete this section on the instructor-provided container-capable workspace to claim a verified image build.


In [ ]:
export LAB_RUN_CONTAINERS=${LAB_RUN_CONTAINERS:-0}


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_CONTAINERS" == 1 ]]; then
    docker info > docker-info.txt
    cd "$WORKSPACE"
    task containers:build VERSION=1.0.0
    for component in crop norm-diff otsu stac; do
        image="ghcr.io/eoap/waterbodies-$component:1.0.0"
        docker image inspect "$image"
        docker run --rm "$image" waterbodies "$component" --help
    done
else
    printf 'Container build skipped: set LAB_RUN_CONTAINERS=1 with a working Docker engine.\n'
fi
)


## Verify the local declarations

These checks run in every workspace and verify release identities, rejection of `latest` and bundled evidence integrity. They do not establish that the optional Docker build ran. Record image IDs and available repository digests separately when you complete that section.


In [ ]:
(
set -euo pipefail
python -m pytest "$WORKSPACE/tests/supply_chain"
)


## Record your findings

**Expected outcome:** All four dependencies use release versions; on a Docker-capable workspace their images build, inspect and run help successfully. Record any skipped section explicitly.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/08-containerize/solution.md) after the activity. Return to [the lab index](README.md).
